In [ ]:
print("hi")

hi


In [ ]:
!pip install -q -U langgraph langchain langchain-groq langchain-community lancedb sentence-transformers torch pandas colorama

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-readers-file 0.5.6 requires pandas<3,>=2.0.0, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.


In [ ]:
!pip install pandas==2.2.2 requests==2.32.4

  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.1
    Uninstalling pandas-3.0.1:
      Successfully uninstalled pandas-3.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
!pip install -q -U langchain-groq

In [ ]:
from google.colab import userdata
import os

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel
from typing import List, Optional
import lancedb
import re
import requests
import torch
import random
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from colorama import Fore, Style

In [ ]:
!pip install -q -U lancedb

In [ ]:
from lancedb.pydantic import LanceModel, Vector

In [ ]:
llm = ChatGroq(model = "llama-3.3-70b-versatile" , temperature = 0.7)


In [ ]:
pip install lancedb

In [ ]:
import torch
import lancedb
from lancedb.pydantic import LanceModel, Vector
!pip install llama-index llama-index-embeddings-huggingface
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
from lancedb.embeddings import get_registry
import requests

In [ ]:
import re
# embedding model
embedding_model = get_registry().get("sentence-transformers").create(
    name="BAAI/bge-small-en-v1.5",
    device="cuda" if torch.cuda.is_available() else "cpu")

"""
name is embedding model
device automatically chooses gpu if available

"""

class Policy(LanceModel):
  text: str = embedding_model.SourceField()
  vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()
"""
text -> source field for embedding model
vector -> 384 dim vector
"""

class VectorStoreRetriever:
  def __init__(self, db_path, docs=""):
    self.db = lancedb.connect("./lancedb")
    table_name = "company_policy"
    if table_name not in self.db.table_names():
        self.table = self.db.create_table(
            table_name,
            schema=Policy,
            mode="create")
    else:
        self.table = self.db.open_table(table_name)

    self.table.add(
        [{"text" : txt.strip()}
         for txt in re.split(r"(?=\n##)", docs)
         if txt.strip()])

  def query(self, query, k=5):
    results = self.table.search(query).limit(k).to_list()
    return [{"page_content" : item["text"],
             "simmilarity" : 1 - item["_distance"]}
            for item in results]

response = requests.get("https://storage.googleapis.com/benchmarks-artifacts/travel-db/swiss_faq.md")
faq_text = response.text
retriever = VectorStoreRetriever(db_path="./lancedb", docs=faq_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_21448/1501709484.py:25: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name not in self.db.table_names():


In [ ]:
pip install llama-index-embeddings-huggingface sentence-transformers torch

In [ ]:
from pydantic import BaseModel
from typing import List, Optional, Annotated
from typing_extensions import TypedDict
from langgraph.graph import add_messages

class Email(BaseModel):
  id : str
  sender : str
  subject : str
  body : str
  final_reply : str = ""
  status : str = "pending"
  failure_reason : str = ""

class EmailState(BaseModel):
  emails : Annotated[List[Email], "List of processed emails"] = []
  processed_emails : List[Email] = []
  current_email : Optional[Email] = None
  current_email_id: str | None = None # Added default value for current_email_id
  policy_context : Optional[str] = ""
  draft : str = ""
  trials : int = 0
  allowed_trials : int = 3
  sendable : bool = False
  exit : bool = False

In [ ]:
print("generate_dummy_emails" in globals())
print(globals().get("generate_dummy_emails"))

False
None


In [ ]:
from faker import Faker
def generate_dummy_emails(num = 3):
  dummy_data = [
        {"id": "1", "sender": "customer1@example.com", "subject": "Flight Delay Compensation", "body": "My flight was delayed by 4 hours. Am I eligible for compensation?"},
        {"id": "2", "sender": "customer2@example.com", "subject": "Baggage Lost", "body": "I lost my baggage on flight SW123. What should I do?"},
        {"id": "3", "sender": "customer3@example.com", "subject": "Refund Request", "body": "I want to cancel my ticket and get a refund."},
    ]
  return [Email(**random.choice(dummy_data)) for _ in range(num)]

In [ ]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.9 MB/s eta 0:00:00


In [ ]:
@tool
def lookup_policy(subject: str, body: str)-> str:
  """Looks up company policy based on subject and body of an email."""
  query = f"{subject} {body}"
  results = retriever.query(query, k=2)
  return "\n\n".join([res["page_content"] for res in results])

def draft_email(subject: str,body: str, policy_context: str)-> str:
  prompt = PromptTemplate.from_template(
      "Draft a professional customer support response for subject: {subject}, body: {body}. Use policy: {policy}.")

  chain = prompt|llm
  response = chain.invoke({"subject" : subject, "body" : body, "policy" : policy_context})
  return response.content

def _validate_draft_content(original_body: str, draft: str)-> bool:
  prompt = PromptTemplate.from_template(
      "Is this draft complete and appropriate? Original: {original}. Draft: {draft}. Reply YES or NO only.")
  chain = prompt|llm
  response = chain.invoke({"original" : original_body, "draft" : draft})
  # Clean the LLM's response to be a valid boolean
  cleaned_response = response.content.strip().strip('.').lower()
  return cleaned_response == 'yes'

In [ ]:
# graph nodes
def fetch_emails(state: EmailState)-> EmailState:
  emails_list = generate_dummy_emails(3)
  print(Fore.GREEN + "Fetched emails." + Style.RESET_ALL)
  return {"emails": emails_list}

def pop_email(state: EmailState)-> EmailState:
  if state.emails:
    state.current_email = state.emails.pop(0)
    state.policy_context = lookup_policy.invoke({"subject": state.current_email.subject, "body": state.current_email.body})
    print(Fore.BLUE + f"processing email: {state.current_email.subject}" + Style.RESET_ALL)
  else:
    state.exit = True
  return state

# Removed the first, redundant draft_email_node definition

def draft_email_node(state: EmailState)-> EmailState:
  # Assuming current_email and policy_context are already set by pop_email
  if state.current_email:
    state.draft = draft_email(state.current_email.subject, state.current_email.body, state.policy_context)
    state.trials += 1
    print(Fore.YELLOW + f"draft attempt { state.trials} : {state.draft[:100]}..." + Style.RESET_ALL)
  return state

def validate_draft(state: EmailState)-> EmailState:
  if state.current_email and state.draft:
    state.sendable = _validate_draft_content(state.current_email.body, state.draft)
  return state


def send_or_skip_email(state: EmailState)-> EmailState:
  if state.current_email:
    if state.sendable:
      state.current_email.final_reply = state.draft
      state.current_email.status = "sent"
      print(Fore.GREEN + f"Sent : {state.draft}" + Style.RESET_ALL)
    else:
      state.current_email.status = "failed"
      state.current_email.failure_reason = "invalid draft"
      print(Fore.RED + f"Failed to send : {state.draft}" + Style.RESET_ALL)
    state.processed_emails.append(state.current_email)
  state.current_email = None
  state.draft = ""
  state.trials = 0
  return state

In [ ]:
memory = MemorySaver()

workflow = StateGraph(state_schema=EmailState)

workflow.add_node("fetch_emails", fetch_emails)
workflow.add_node("pop_email", pop_email)
workflow.add_node("draft_email", draft_email_node)
workflow.add_node("validate_draft", validate_draft)
workflow.add_node("send_or_skip_email", send_or_skip_email)

workflow.add_edge(START, "fetch_emails")
workflow.add_edge("fetch_emails", "pop_email")


def has_email(state: EmailState):
  return "draft_email" if state.current_email else END
workflow.add_conditional_edges("pop_email", has_email)

workflow.add_edge("draft_email", "validate_draft")

def route_after_validation(state: EmailState):
  if state.sendable:
    return "send_or_skip_email"
  elif state.trials >= state.allowed_trials:
    return "send_or_skip_email"
  else:
    return "draft_email"

workflow.add_conditional_edges("validate_draft", route_after_validation)

workflow.add_edge("send_or_skip_email", "pop_email")

graph = workflow.compile(checkpointer = memory)

In [ ]:
initial_state = EmailState()
config = {"configurable": {"thread_id": "customer_support_thread_1"}}

print(Fore.GREEN + "starting autonomous customer support agent.." + Style.RESET_ALL)


for output in graph.stream(initial_state, config):
  for node, state_update in output.items():
    print(Fore.CYAN + f"Node completed ; {node}" + Style.RESET_ALL)

final_state = graph.get_state(config).values
print("\n processed emails:")
for email in final_state['processed_emails']:
  print(f"- ID: {email.id}, Status: {email.status}, Reply: {email.final_reply[:100]}...")

starting autonomous customer support agent..
Fetched emails.
Node completed ; fetch_emails
processing email: Baggage Lost
Node completed ; pop_email
draft attempt 1 : Subject: Re: Baggage Lost on Flight SW123

Dear [Customer's Name],

Thank you for reaching out to us...
Node completed ; draft_email
Node completed ; validate_draft
draft attempt 2 : Subject: Re: Baggage Lost on Flight SW123

Dear [Customer's Name],

Thank you for reaching out to us...
Node completed ; draft_email
Node completed ; validate_draft
Sent : Subject: Re: Baggage Lost on Flight SW123

Dear [Customer's Name],

Thank you for reaching out to us regarding the loss of your baggage on flight SW123. We apologize for the inconvenience this has caused and are here to assist you.

To report your lost baggage, we recommend that you contact our baggage services team directly. They will be able to guide you through the process of reporting your lost baggage and will do their best to locate it as soon as possible.

In the mea

draft attempt 1 : Subject: Re: Flight Delay Compensation

Dear [Customer Name],

Thank you for reaching out to us rega...
Node completed ; draft_email
Node completed ; validate_draft
Sent : Subject: Re: Flight Delay Compensation

Dear [Customer Name],

Thank you for reaching out to us regarding your recent flight delay. We apologize for the inconvenience this has caused and appreciate your patience and understanding.

To determine your eligibility for compensation, we would like to inform you that our airline operates under the EU Flight Delay Compensation Regulation (EC 261/2004). According to this regulation, if your flight was delayed by 3 hours or more, you may be eligible for compensation.

Since your flight was delayed by 4 hours, we would like to investigate this matter further. To proceed with your claim, could you please provide us with the following information:

* Your booking reference number
* Your flight number and travel dates
* A detailed description of the delay, inclu